# LoRA 推論實戰：從 Adapter 載入到模型合併

## 學習目標

完成本 notebook 後，你將能夠：

1. 以 2026 統一慣例（`device_map='auto'` + `torch_dtype=torch.bfloat16` + `use_safetensors=True`）載入基礎模型
2. 用 `PeftModel.from_pretrained()` 掛載 LoRA adapter，不修改基礎模型權重
3. 以 `tokenizer.apply_chat_template()` 建構可跨模型移植的對話 prompt，取代脆弱的手刻 `Human:/Assistant:` 字串
4. 以 `merge_and_unload()` 將 adapter 權重永久融合進基礎模型，取得零 PEFT overhead 的推論模型
5. 以 `save_pretrained(safe_serialization=True)` 儲存為安全的 safetensors 格式

## 前置條件

- 已完成 `chatbot_lora.ipynb`，並在 `./chatbot/checkpoint-<N>/` 存有訓練好的 LoRA adapter
- GPU 建議 >= 8 GB VRAM（此 notebook 使用 ~1.4B 參數模型，bf16 約需 3 GB）

## 與相鄰 notebook 的銜接

| 上游 | 本 notebook | 下游 |
|---|---|---|
| [`chatbot_lora.ipynb`](chatbot_lora.ipynb) — LoRA 訓練，產生 adapter checkpoint | **lora_inference.ipynb** — 推論、合併、儲存 | [`../02-IA3/chatbot_ia3.ipynb`](../02-IA3/chatbot_ia3.ipynb) — IA3 另一種 PEFT 方法 |

## 版本鎖定

確保環境與本 notebook 預期的 API 版本一致。
請使用 `transformers>=4.46`、`peft>=0.13`，以確保 `BitsAndBytesConfig` 量化 API 與 `PeftModel` safetensors 序列化均可正常運作。

In [ ]:
# Version pinning — run once per environment
# %pip install -q \
#     "transformers>=4.46" \
#     "peft>=0.13" \
#     "accelerate>=1.0" \
#     "safetensors>=0.4" \
#     "torch>=2.4"

import importlib, sys

REQUIRED = {
    "transformers": "4.46",
    "peft": "0.13",
    "accelerate": "1.0",
    "safetensors": "0.4",
}

for pkg, min_ver in REQUIRED.items():
    ver = importlib.import_module(pkg).__version__
    ok = tuple(int(x) for x in ver.split(".")[:2]) >= tuple(int(x) for x in min_ver.split(".")[:2])
    status = "OK" if ok else f"OUTDATED (have {ver}, need >={min_ver})"
    print(f"{pkg:>16} {ver:>10}  {status}")

## 1. 匯入套件

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

## 2. 載入基礎模型

### 2026 統一慣例

```python
model = AutoModelForCausalLM.from_pretrained(
    "Langboat/bloom-1b4-zh",
    device_map="auto",        # 自動分配 GPU/CPU/disk
    torch_dtype=torch.bfloat16,  # 直接以 bf16 載入
    use_safetensors=True,     # 優先讀 .safetensors
)
```

**為什麼 bf16 優於 fp16/fp32？**

- `fp32`：VRAM 佔用最大（每參數 4 bytes），推論幾乎不必要
- `fp16`：VRAM 減半，但動態範圍窄（最大值 65504），梯度更新容易溢位；推論尚可但訓練有風險
- `bf16`：與 fp32 相同指數位元（動態範圍大），尾數較少但推論精度足夠；A100/H100 bf16 吞吐與 fp16 相同；RTX 30/40 系列亦支援

**`device_map='auto'` 的語意**

HuggingFace `accelerate` 依序優先使用 GPU → CPU RAM → disk。1.4B bf16 約 3 GB，單 GPU 通常全進 GPU。若 VRAM 不足，accelerate 自動分層卸載，不必手動計算 `.to(device)`。

**safetensors 相對 pickle（`.bin`）的優勢**

- 零反序列化安全漏洞（pickle 可執行任意 Python）
- mmap 直接映射，載入速度快 2-3x
- 支援懶惰載入（只讀需要的 tensor）

In [ ]:
BASE_MODEL_ID = "Langboat/bloom-1b4-zh"
# VRAM note: bloom-1b4 in bf16 ~= 3 GB. If OOM, try torch_dtype=torch.float16
# or add quantization_config=BitsAndBytesConfig(load_in_4bit=True, ...) (see 04-kbits-tuning)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

print(f"Model dtype : {next(model.parameters()).dtype}")
print(f"Device map  : {model.hf_device_map}")

## 3. 掛載 LoRA Adapter

`PeftModel.from_pretrained()` 的語意：

- 讀取 adapter 目錄中的 `adapter_config.json`，重建 `LoraConfig`，並在基礎模型對應層注入低秩矩陣 A/B
- **基礎模型權重完全不動**；推論時的權重是 `W + ΔW`，其中 `ΔW = B × A × α/r`
- `adapter_weights` 僅幾 MB（視 `r` 與目標模組數量）；base model 不必重複儲存

`PeftConfig.from_pretrained()` 可先讀取 adapter 的設定，用於驗證 base model id 是否吻合。

In [ ]:
ADAPTER_PATH = "./chatbot/checkpoint-180"  # adjust to your actual checkpoint directory

# Verify adapter config before loading
peft_config = PeftConfig.from_pretrained(ADAPTER_PATH)
print(f"Base model expected : {peft_config.base_model_name_or_path}")
print(f"LoRA rank (r)       : {peft_config.r}")
print(f"LoRA alpha          : {peft_config.lora_alpha}")
print(f"Target modules      : {peft_config.target_modules}")

p_model = PeftModel.from_pretrained(model, model_id=ADAPTER_PATH)
print("\nAdapter loaded.")
print(p_model)

## 4. 以 LoRA Adapter 推論

### 4.1 建構對話 Prompt

2026 統一做法：使用 `tokenizer.apply_chat_template()` 建構輸入。

```python
messages = [{"role": "user", "content": question}]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
ipt = tokenizer(prompt, return_tensors="pt")
```

**為什麼用 `apply_chat_template()`？**

- 每個模型的特殊 token 不同：Llama 3 用 `<|start_header_id|>`、Mistral 用 `[INST]`、ChatGLM 有自己的 `build_chat_input`。手刻格式字串在換模型時全部失效
- `apply_chat_template` 從 tokenizer 的 `chat_template` 欄位讀取 Jinja2 模板，格式由模型作者維護，你的程式碼不需要改
- 訓練端與推論端用同一模板，確保分佈一致（訓練/推論 mismatch 是常見的微調效果差的原因）
- 2026 多模態訊息（image/audio token）也使用同一套 `messages` 結構，是通往 05-Multimodal 的關鍵橋樑

**注意**：BLOOM 系列的 tokenizer 可能未內建 `chat_template`。此時 `apply_chat_template` 會使用 HuggingFace 的預設 fallback 模板。若你自訂了訓練格式，可在 tokenizer 上設定 `tokenizer.chat_template = your_jinja2_str`，以確保一致性。

In [ ]:
def build_prompt(tokenizer: AutoTokenizer, user_message: str) -> dict:
    """Build tokenized input using chat_template when available, with graceful fallback."""
    messages = [{"role": "user", "content": user_message}]

    if tokenizer.chat_template is not None:
        # 2026 standard path: use model-defined Jinja2 template
        prompt_str = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        # Fallback for models without a registered chat_template
        # (e.g. older BLOOM checkpoints)
        # NOTE: this should match the format used during LoRA training
        prompt_str = f"Human: {user_message}\n\nAssistant: "

    return tokenizer(prompt_str, return_tensors="pt")


question = "考试有哪些技巧？"
ipt = build_prompt(tokenizer, question)

# Move inputs to the same device as the model
device = next(p_model.parameters()).device
ipt = {k: v.to(device) for k, v in ipt.items()}

print(f"Prompt (decoded): {tokenizer.decode(ipt['input_ids'][0], skip_special_tokens=False)}")

### 4.2 生成回覆（帶 LoRA adapter）

In [ ]:
# Greedy decoding; max_new_tokens prevents runaway generation
with torch.inference_mode():
    output_ids = p_model.generate(
        **ipt,
        do_sample=False,
        max_new_tokens=200,
        repetition_penalty=1.1,  # mild penalty to reduce repetition common in BLOOM
    )

# Decode only the newly generated tokens (exclude prompt)
generated = output_ids[0][ipt["input_ids"].shape[-1]:]
response = tokenizer.decode(generated, skip_special_tokens=True)

print(f"Question : {question}")
print(f"Response : {response}")

## 5. 模型參數合併（merge_and_unload）

### 為什麼要合併？

帶 adapter 的 `PeftModel` 在推論時的計算路徑：

```
output = W(x) + B(A(x)) * (alpha / r)   # 兩次矩陣乘法
```

`merge_and_unload()` 把 `ΔW = B × A × (alpha/r)` 靜態加到 `W` 上，之後推論只需：

```
output = (W + ΔW)(x)                     # 一次矩陣乘法
```

**好處**：
- 推論速度與純 base model 相同，無額外 overhead
- 模型可以直接以標準 `AutoModelForCausalLM` 載入，無需安裝 PEFT
- 可進一步做量化（BitsAndBytesConfig）或部署到不支援 PEFT 的環境

**代價**：
- 合併後無法再分離 adapter；若要繼續訓練 LoRA，要保留 un-merged checkpoint
- 合併模型大小 = 原 base model 大小（adapter 的節省消失了）

In [ ]:
# merge_and_unload() returns a standard nn.Module (no longer a PeftModel)
merge_model = p_model.merge_and_unload()
print(type(merge_model))
print(merge_model)

### 5.1 驗證合併後推論結果一致

In [ ]:
# Re-build and move inputs for merged model
ipt_merged = build_prompt(tokenizer, question)
device_merged = next(merge_model.parameters()).device
ipt_merged = {k: v.to(device_merged) for k, v in ipt_merged.items()}

with torch.inference_mode():
    output_ids_merged = merge_model.generate(
        **ipt_merged,
        do_sample=False,
        max_new_tokens=200,
        repetition_penalty=1.1,
    )

generated_merged = output_ids_merged[0][ipt_merged["input_ids"].shape[-1]:]
response_merged = tokenizer.decode(generated_merged, skip_special_tokens=True)

print(f"Question        : {question}")
print(f"Response (PEFT) : {response}")
print(f"Response (merged): {response_merged}")
print(f"\nOutputs match: {response == response_merged}")

## 6. 儲存合併模型

以 `safe_serialization=True` 明確輸出 safetensors 格式：

```python
merge_model.save_pretrained(
    "./chatbot/merge_model",
    safe_serialization=True,   # 輸出 .safetensors 而非 .bin
)
```

`safe_serialization=True` 在 transformers >= 4.36 已是預設值，但明確寫出來可作為自我文件，也方便未來 diff 確認行為未改變。

In [ ]:
SAVE_PATH = "./chatbot/merge_model"

merge_model.save_pretrained(SAVE_PATH, safe_serialization=True)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Merged model saved to: {SAVE_PATH}")

# Verify the output files
import os
for f in sorted(os.listdir(SAVE_PATH)):
    size_mb = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1024**2
    print(f"  {f:<45} {size_mb:>8.2f} MB")

## 7. 從儲存的合併模型重新載入驗證

這是部署前的完整性驗證：以純 `AutoModelForCausalLM`（無 PEFT）重新載入並推論，確認儲存格式正確、不依賴 PEFT 環境。

In [ ]:
# Load saved merged model as a standard causal LM — no PeftModel needed
loaded_model = AutoModelForCausalLM.from_pretrained(
    SAVE_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
loaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

ipt_loaded = build_prompt(loaded_tokenizer, question)
device_loaded = next(loaded_model.parameters()).device
ipt_loaded = {k: v.to(device_loaded) for k, v in ipt_loaded.items()}

with torch.inference_mode():
    output_ids_loaded = loaded_model.generate(
        **ipt_loaded,
        do_sample=False,
        max_new_tokens=200,
        repetition_penalty=1.1,
    )

generated_loaded = output_ids_loaded[0][ipt_loaded["input_ids"].shape[-1]:]
response_loaded = loaded_tokenizer.decode(generated_loaded, skip_special_tokens=True)

print(f"Response (reloaded): {response_loaded}")
print(f"Round-trip consistent: {response_merged == response_loaded}")

## 小結

本 notebook 示範了 LoRA 推論的完整流程：

| 步驟 | 2026 統一慣例 |
|---|---|
| 載入基礎模型 | `device_map='auto'`, `torch_dtype=bfloat16`, `use_safetensors=True` |
| 對話 prompt | `tokenizer.apply_chat_template(messages, ...)` |
| 輸入移裝置 | `{k: v.to(device) for k, v in ipt.items()}` |
| 儲存 | `save_pretrained(safe_serialization=True)` → `.safetensors` |

**關鍵概念回顧**：

- LoRA adapter 不修改基礎模型，讓「同一個 base + 多個 adapter」的用法成為可能
- `merge_and_unload()` 消除推論時的 ΔW 計算 overhead；代價是失去 adapter 可分離性
- `apply_chat_template` 是推論與訓練格式一致的唯一可靠保證；不要手刻 prompt 格式
- safetensors 格式在安全性與載入速度上均優於 pickle；2026 起所有新模型預設此格式

## 練習

1. **換問題測試**：修改 `question` 變數，觀察合併前後的回覆是否完全一致（`do_sample=False` 下應該一模一樣）。

2. **加入 chat_template**：在 `tokenizer` 上設定一段自訂 Jinja2 `chat_template`，使其格式與訓練時的 `Human:/Assistant:` 相同，並確認 `build_prompt` 函數的 fallback 分支不再被觸發。

3. **多輪對話**：修改 `build_prompt`，傳入一個含多個 `user`/`assistant` 回合的 `messages` list，觀察模型在多輪情境下的回覆品質。

4. **adapter 疊加**：訓練兩個不同任務的 LoRA adapter（如中文 QA 與程式碼補全），嘗試以 `set_adapter` 動態切換，或以 `add_adapter` + `enable_adapters_for_layers` 組合使用，觀察效果差異。

5. **量化推論**：在載入 `merge_model` 後，嘗試加入 `BitsAndBytesConfig(load_in_4bit=True, ...)` 進一步壓縮 VRAM 用量（參見 `04-kbits-tuning` 模組）。